In [11]:
from pathlib import Path
from typing import List, Dict, Any
from tqdm import tqdm
from pypdf import PdfReader
import pandas as pd
import numpy
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # keep ChromaDB fully offline (no usage telemetry)
from docx import Document
import re
import warnings
import ollama
from langchain_core.documents import Document as LCDocument 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama

DATA_DIR = Path("data")
DB_DIR = "db"
COLLECTION_NAME = "policies"
DATA_DIR.mkdir(exist_ok=True)
pd.options.display.max_colwidth = 200
pd.options.display.max_seq_items = 200
pd.set_option('display.max_rows', None)
warnings.filterwarnings("ignore")


In [12]:
def load_txt(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")

def load_docx(path: Path) -> str:
    doc = Document(str(path))
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

def load_pdf(path: Path) -> str:
    reader = PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        if text.strip():
            pages.append(text)
    return "\n".join(pages)

def load_documents(data_dir: Path) -> List[Dict[str, Any]]:
    docs = []
    for p in data_dir.rglob("*"):
        if p.is_dir():
            continue
        ext = p.suffix.lower()
        if ext not in [".pdf", ".docx", ".txt"]:
            continue

        if ext == ".pdf":
            text = load_pdf(p)
        elif ext == ".docx":
            text = load_docx(p)
        else:
            text = load_txt(p)

        if text.strip():
            docs.append({"text": text, "source": p.name, "path": str(p)})
    return docs


In [13]:
raw_docs = load_documents(DATA_DIR)
len(raw_docs), [d["source"] for d in raw_docs[:5]]

(1, ['HR-Master-Document.pdf'])

In [14]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

lc_docs = []
for d in raw_docs:
    chunks = splitter.split_text(d["text"])
    for idx, chunk in enumerate(chunks):
        lc_docs.append(
            LCDocument(
                page_content=chunk,
                metadata={"source": d["source"], "path": d["path"], "chunk": idx},
            )
        )

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Start from an empty collection so re-running this cell doesn't add duplicate chunks
Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
    embedding_function=embeddings,
).delete_collection()

vectordb = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
    embedding_function=embeddings,
)

batch_size = 200
for i in tqdm(range(0, len(lc_docs), batch_size), desc="Indexing"):
    vectordb.add_documents(lc_docs[i:i+batch_size])

vectordb.persist()
print(f"✅ Indexed {len(lc_docs)} chunks into ./{DB_DIR}")


Indexing: 100%|██████████████████████████████████████████████████████████████████████████| 1/1 [00:34<00:00, 34.79s/it]

✅ Indexed 12 chunks into ./db


In [15]:
SYSTEM_RULES = """You answer questions using ONLY the provided policy excerpts.
If the answer is not in the excerpts, say: "I couldn't find that in the provided documents."
Be precise. If a rule has conditions/exceptions, include them.
Always include citations as: (source, chunk).
"""

llm = Ollama(model="llama3.1:8b", temperature=0.2)

def format_context(hits):
    out = []
    for h in hits:
        src = h.metadata.get("source")
        ch = h.metadata.get("chunk")
        out.append(f"[{src} | chunk {ch}] {h.page_content}")
    return "\n\n".join(out)

def ask(question: str, k: int = 5):
    retriever = vectordb.as_retriever(search_kwargs={"k": k})

    hits = retriever.invoke(question)

    context = format_context(hits)

    prompt = f"""{SYSTEM_RULES}

    Question: {question}

    Policy excerpts:
    {context}

    Answer (with citations):
    """
    return llm.invoke(prompt)

In [16]:
print(ask("What's the resignation notice period?"))

I couldn't find any information about the resignation notice period in the provided policy excerpts.


In [17]:
print(ask("How many leaves each year?"))

I couldn't find any information about the number of leaves each year in the provided policy excerpts.


In [18]:
print(ask("What are the things that I should not do in the workplace?"))

I couldn't find any specific policy excerpts that outline the things that you should not do in the workplace. However, I can suggest that you refer to the "Employee Disciplinary Policy" (chunk 4) and the "Code of Conduct Policy" (chunk 4) for guidance on expected behavior and consequences of misconduct.

(Employee Disciplinary Policy, chunk 4)
(Code of Conduct Policy, chunk 4)


In [19]:
print(ask("What is the policy for health insurace?"))

I couldn't find any information about health insurance in the provided policy excerpts.


In [20]:
print(ask("What is the restriction about the computer usage?"))

I couldn't find any information about computer usage restrictions in the provided policy excerpts.
